# ZH - Mesterséges intelligencia - Dealer detector

### Szabályok

- A rendelkezésre álló idő: **85 perc**, **+5 perc** feltöltés
- Egyéni munka, mindenki önállóan dolgozik (AI és egyéb humán segítség nélkül)
- A korábbi órák anyagai használhatóak a laborvezetővel egyeztetett módon
- A munkafüzetben megjelölt mezőkben dolgozzon (`#TODO`), de új cellákat is felvehet, igény szerint

__Beadás__ (http://zh.nik.lan): A kitöltött, elmentett (!) notebook, futási eredményekkel
- A _warning_-ok figyelmen kívül hagyhatóak

### Feladat: 
Készítsen osztályozó modellt, amely képes:
- személygépjárművek eladási adatai alapján megjósolni, hogy az autót magánszemély vagy kereskedő árulja-e (célváltozó: `seller_type`)
- __hipotézisünk__: a kereskedők drágábban árulják az azonos kategóriájú (évjárat, köbcenti, futott kilóméter, stb.) autókat

### 0. LÉPÉS: Alapvető könyvtárak importálása

(További könyvtárak importálására is szükség lehet.)


In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### 1. LÉPÉS: Adat betöltése

- Töltse be az autók adatait tartalmazó `Car details v3.csv` fájlt egy `df` elnevezésű DataFrame objektumba
- Jelenítse meg az első három sort


In [52]:
# TODO
df = pd.read_csv('Car details v3.csv')
df.head(3)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0


### 2. LÉPÉS: Adatok áttekintése

Jelenítse meg, hogy: 
- hány sor és hány oszlop található a DataFrame-ben
- az egyes oszlopoknak mi a típusa
- a szám típusú oszlopoknak mik az értéktartományai (minimum, maximum érték)
- a `seller_type` kategorikus oszlopnak mi az értékkészlete

In [53]:
# TODO
print(df.shape)
df.info()
print(df.describe())
print(df['seller_type'].unique())
print(df['owner'].unique())
print(df['transmission'].unique())
print(df['fuel'].unique())

(8128, 13)
<class 'pandas.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   str    
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   str    
 5   seller_type    8128 non-null   str    
 6   transmission   8128 non-null   str    
 7   owner          8128 non-null   str    
 8   mileage        7907 non-null   str    
 9   engine         7907 non-null   str    
 10  max_power      7913 non-null   str    
 11  torque         7906 non-null   str    
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), str(9)
memory usage: 825.6 KB
              year  selling_price     km_driven        seats
count  8128.000000   8.128000e+03  8.128000e+03  7907.000000
mean   2013.804011   6.382718e+05  6.981951e+04    

### 3. LÉPÉS: Adattisztítás

- jelenítse meg, hogy az melyik oszlop hány üres (NaN) cellát tartalmaz
- minden olyan sort távolítson el, amelynek valamely attribútuma (cellája) üres
- az eltávolított sorok számát tárolja el egy removed_records változóban, amit írjon ki


In [54]:
# TODO
print(df.isna().sum())

prev_rows = df.shape[0]
df = df.dropna()

removed_records = prev_rows - df.shape[0]
print("removed:", removed_records)

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64
removed: 222


### 4. LÉPÉS: Feature engineering

- a `transmission` oszolopot képezze le 1 (kézi váltós) és 2 (automata) értékekre
- az `engine` oszlopot alakítsa mértékegység (`CC`) nélküli, numerikus adattá
- az `owner` oszlop képezze le számokra: 0: teszt autó, 1, ..., 4: tulajdonosok száma (4: négy vagy több)
- a `seller_type` oszlop értéke legyen 0: ha magánszemély az eladó, különben 1 
- a `fuel` oszlopot bontsa fel 1-hot-enconding alapján 4 oszlopra (CNG, Diesel, ...)
- törölje a `name`, `mileage`, `engine`, `max_power`, `torque` oszlopokat
- ellenőrizze, hogy most már minden oszlop numerikus típusú-e

In [55]:
# TODO

# hipotezis: year, selling_price, km_driven, fuel, transmission, owner, seats, => price, seller

df['transmission'] = df['transmission'].apply(lambda x: 2 if x == 'Automatic' else 1) 

# engine, "1248 CC"
df['engine'] = df['engine'].str.replace("CC", "")
df['engine'] = df['engine'].astype(float)

mapping = {
    'First Owner': 1,
    'Second Owner': 2,
    'Third Owner': 3,
    'Fourth & Above Owner': 4,
    'Test Drive Car': 0
}
df['owner'] = df['owner'].map(mapping)

df['seller_type'].unique() # ['Individual', 'Dealer', 'Trustmark Dealer']
mapping = {
    'Individual': 0,
    'Dealer': 1,
    'Trustmark Dealer': 1
}
df['seller_type'] = df['seller_type'].map(mapping)
#vagy
#df['seller_type'] = df['seller_type'].apply(lambda x: 0 if x == 'Individual' else 1)

df = pd.get_dummies(df, columns=['fuel']) # fuel oszlop helyett 4

df = df.drop(columns=['name', 'mileage', 'engine', 'max_power', 'torque'])

df.info()

<class 'pandas.DataFrame'>
Index: 7906 entries, 0 to 8127
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   year           7906 non-null   int64  
 1   selling_price  7906 non-null   int64  
 2   km_driven      7906 non-null   int64  
 3   seller_type    7906 non-null   int64  
 4   transmission   7906 non-null   int64  
 5   owner          7906 non-null   int64  
 6   seats          7906 non-null   float64
 7   fuel_CNG       7906 non-null   bool   
 8   fuel_Diesel    7906 non-null   bool   
 9   fuel_LPG       7906 non-null   bool   
 10  fuel_Petrol    7906 non-null   bool   
dtypes: bool(4), float64(1), int64(6)
memory usage: 525.0 KB


### 5. LÉPÉS: Modell inputok, outputok előkészítése

- válassza le az adatokat a célváltozó (`seller_type`) halmazáról: előbbi legyen `X`, az utóbbi `y` elnevezésű
- ossza fel az `X` és `y` halmazokat tanító és teszt részhalmazra 80%-20% arányban
- ezekre használja a`X_train`, `y_train`, `X_test`, `y_test` változóneveket
- állítson be egy `random_state` értéket, a kísérlet megismételhetősége érdekében
- használja a `stratify=y` opciót, hogy a tanító és teszthalmazokban arányosan legyenek különböző kimenetelek

In [ ]:
# TODO
from sklearn.model_selection import train_test_split

X = df.drop(columns=["seller_type"])
y = df["seller_type"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # a stratify egyenlo aranyban osztja el az adatokat a test es a trainben hogy nehogy pl ne legyen a testben egy manualis auto sem

### 6. LÉPÉS: Modell tanítása

- hozzon létre egy __Döntési fa__ osztályozó modellt
- a fa maximális mélysége 20 legyen
- használja a __model1__ változónevet ehhez a modellhez
- állítson be egy random_state értéket, a kísérlet megismételhetősége érdekében
- tanítsa be

In [57]:
# TODO
from sklearn.tree import DecisionTreeClassifier
model1 = DecisionTreeClassifier(max_depth=20, random_state=42)
model1.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

### 7. LÉPÉS: Modell kiértékelése

- értékelje ki a modellt a __teszt__ halmazon a következő metrikákkal:
    - pontosság (accuracy)
    - precizitás (precision)
    - szenzitivitás (recall)
    - igazságmátrix (confusion matrix)

In [58]:
# TODO
y_pred = model1.predict(X_test)
print(classification_report(y_pred, y_test))
print(confusion_matrix(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1311
           1       0.69      0.69      0.69       271

    accuracy                           0.89      1582
   macro avg       0.81      0.81      0.81      1582
weighted avg       0.89      0.89      0.89      1582

[[1228   83]
 [  85  186]]


### 8. LÉPÉS: Második modell

- válasszon és tanítson be ugyanezen adatokon egy másik, **lineáris osztályozó modellt** (ne használjon SVM-et, a hosszú futási idő miatt, warning esetén fontolja meg a `max_iter` érték növelését)
- használja a __model2__ változónevet ehhez a modellhez
- állítson be egy random_state értéket, a kísérlet megismételhetősége érdekében

In [59]:
# TODO
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

# model2 = SVC(kernel="linear", random_state=42, max_iter=10)
model2 = LogisticRegression(random_state=42, max_iter=1000)
model2.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

### 9. LÉPÉS: Második modell kiértékelése

- értékelje ki a második modell teljesítményét a teszt halmazon a következő metrikákkal:
    - pontosság (accuracy)
    - precizitás (precision)
    - szenzitivitás (recall)
    - igazságmátrix (confusion matrix)

In [60]:
# TODO
y_pred = model2.predict(X_test)
print(classification_report(y_pred, y_test))
print(confusion_matrix(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.99      0.86      0.92      1522
           1       0.19      0.83      0.30        60

    accuracy                           0.86      1582
   macro avg       0.59      0.84      0.61      1582
weighted avg       0.96      0.86      0.90      1582

[[1303  219]
 [  10   50]]


### 10. LÉPÉS: Modellek összehasonlítása

- melyik modell a megfelelőbb: model1  vagy model2,
- ha az a legfontosabb szempont, hogy ha a modell magánszemély eladót prediktál (0 azonosítójú osztály), akkor jó eséllyel ténylegesen is magánszemély legyen az eladó
- szövegesen indokolja (a vastagított részek kiválasztásával, a konkrét értékek megadása szükséges):
Az eredények alapján a __model1 | model2__ megfelelőbb, mert a __pontosság | precizitás | szenzitivitás | f1__ értéke magasabb (__x.xx > y.yy__).

Az eredények alapján a __model2__ megfelelőbb, 
mert a __precizitás__ értéke (a 0-ás osztályra vonatkozóan) magasabb
(__0.99 > 0.93__).
